# Initialization

In [1]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [3]:
items = pd.read_parquet("items.par")
events = pd.read_parquet("events.par")

# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [4]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()

train_test_global_time_split_idx = events["started_at"] < train_test_global_time_split_date
events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()
# количество пользователей, которые есть и в train, и в test
common_users = users_train[users_train.isin(users_test)]

print(len(users_train), len(users_test), len(common_users)) 

428220 123223 120858


In [5]:
cold_users = users_test[~users_test.isin(users_train)]

print(len(cold_users)) 

2365


In [6]:
from sklearn.preprocessing import MinMaxScaler

# переименуем book_id -> item_id для единообразия
events_train = events_train.rename(columns={"book_id": "item_id"})
events_test = events_test.rename(columns={"book_id": "item_id"})
items = items.rename(columns={"book_id": "item_id"})

top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["item_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)

# сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values("popularity_score", ascending=False)

# выбираем первые 100 айтемов со средней оценкой avg_rating не меньше 4
top_k_pop_items = item_popularity.query("avg_rating >= 4").head(100)

# количество пользователей оценило книгу, попавшую на первое место в top_k_pop_items 
print(top_k_pop_items["users"].iloc[0])

# добавляем информацию о книгах
top_k_pop_items = top_k_pop_items.merge(
    items.set_index("item_id")[["author", "title", "genre_and_votes", "publication_year"]], on="item_id")

# вывод информаци
#with pd.option_context('display.max_rows', 100):
#    display(top_k_pop_items[["item_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]]) 

cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(top_k_pop_items[["item_id", "avg_rating"]], on="item_id", how="left")

cold_user_items_no_avg_rating_idx = cold_users_events_with_recs["avg_rating"].isnull()
cold_user_recs = cold_users_events_with_recs[~cold_user_items_no_avg_rating_idx] \
    [["user_id", "item_id", "rating", "avg_rating"]] 

share = (~cold_users_events_with_recs["avg_rating"].isnull()).mean()
print(round(share, 2))

# посчитаем метрики рекомендаций
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = mean_squared_error(cold_user_recs["rating"], cold_user_recs["avg_rating"], squared=False)
mae = mean_absolute_error(cold_user_recs["rating"], cold_user_recs["avg_rating"])
print(round(rmse, 2), round(mae, 2)) 

# посчитаем покрытие холодных пользователей рекомендациями

cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}") 


20207
0.2
0.78 0.62
Доля пользователей без релевантных рекомендаций: 0.59
Среднее покрытие пользователей: 0.44


# === Знакомство: первые персональные рекомендации

# === Базовые подходы: коллаборативная фильтрация

In [7]:
events = events.rename(columns={"book_id": "item_id"})
ui = events[["user_id", "item_id", "rating"]]  # при необходимости сначала rename book_id -> item_id
n_users = ui["user_id"].nunique()
n_items = ui["item_id"].nunique()
n_events = len(ui)
sparsity = 1 - n_events / (n_users * n_items)
print(round(sparsity, 4))  # 0.9993

0.9993


In [8]:
from surprise import Dataset, Reader
from surprise import SVD
from surprise import accuracy
from surprise import NormalPredictor

# используем Reader из библиотеки surprise для преобразования событий (events)
# в формат, необходимый surprise
reader = Reader(rating_scale=(1, 5))
surprise_train_set = Dataset.load_from_df(events_train[['user_id', 'item_id', 'rating']], reader)
surprise_train_set = surprise_train_set.build_full_trainset()

# инициализируем модель
svd_model = SVD(n_factors=100, random_state=0)

# обучаем модель
svd_model.fit(surprise_train_set) 

surprise_test_set = list(events_test[['user_id', 'item_id', 'rating']].itertuples(index=False))

# получаем рекомендации для тестовой выборки
svd_predictions = svd_model.test(surprise_test_set) 

rmse_svd = accuracy.rmse(svd_predictions)
mae_svd = accuracy.mae(svd_predictions)
                     
print('рекомендации для тестовой выборки: rmse|mae', rmse_svd, mae_svd) 

# инициализируем состояние генератора, это необходимо для получения
# одной и той же последовательности случайных чисел, только в учебных целях
np.random.seed(0)

random_model = NormalPredictor()

random_model.fit(surprise_train_set)
random_predictions = random_model.test(surprise_test_set) 

mae_random = accuracy.mae(random_predictions)

#Рассчитайте значение MAE для random_predictions.
#На сколько процентов MAE для случайных рекомендаций от NormalPredictor выше значения MAE от SVD? Ответ округлите до целых.
print(round((mae_random - mae_svd) / mae_svd * 100))

RMSE: 0.8289
MAE:  0.6474
рекомендации для тестовой выборки: rmse|mae 0.8288711689059135 0.647437483750257
MAE:  1.0018
55


In [9]:
def get_recommendations_svd(user_id, all_items, events, model, include_seen=True, n=5):

    """ возвращает n рекомендаций для user_id """
    
    # получим список идентификаторов всех книг
    all_items = set(events['item_id'].unique())
        
    # учитываем флаг, стоит ли уже прочитанные книги включать в рекомендации
    if include_seen:
        items_to_predict = list(all_items)
    else:
        # получим список книг, которые пользователь уже прочитал ("видел")
        seen_items = set(events[events["user_id"] == user_id]['item_id'].unique())
        
        # книги, которые пользователь ещё не читал
        # только их и будем включать в рекомендации
        items_to_predict = list(all_items - seen_items)
    
    # получаем скоры для списка книг, т. е. рекомендации
    predictions = [model.predict(user_id, item_id) for item_id in items_to_predict]
    
    # сортируем рекомендации по убыванию скора и берём только n первых
    recommendations = sorted(predictions, key=lambda x: x.est, reverse=True)[:n]
    
    return pd.DataFrame([(pred.iid, pred.est) for pred in recommendations], columns=["item_id", "score"])

get_recommendations_svd(1296647, items, events_test, svd_model) 



,item_id,score
0,7864312,4.981188
1,25793186,4.912001
2,12174312,4.898052
3,13208,4.894869
4,33353628,4.891661


### Коллаборативная фильтрация: ALS

In [10]:
import scipy
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"]) # ваш код здесь #
events_test["item_id_enc"] =  item_encoder.transform(events_test["item_id"]) # ваш код здесь #

print(events_train["item_id_enc"].max())

#Вычислите размер матрицы user_item_matrix_train, как если бы она хранила все свои элементы, включая пропуски, и для каждого элемента использовался бы один байт. Ответ приведите в виде целого числа гигабайтов, отбросив дробную часть.
#Подсказка: 
#Умножьте количество строк на количество столбцов, а затем результат разделите на 1024^3

n_rows = len(user_encoder.classes_)      # число пользователей
n_cols = len(item_encoder.classes_)      # число объектов (книг)
size_gb = (n_rows * n_cols) // (1024 ** 3)
print('matrix size: ', size_gb)

# создаём sparse-матрицу формата CSR 
# sparse-формат numpy-матриц позволяет сильно уменьшить требование к размеру памяти
user_item_matrix_train = scipy.sparse.csr_matrix((
    events_train["rating"],
    (events_train['user_id_enc'], events_train['item_id_enc'])),
    dtype=np.int8) 

import sys

size_gb_sparse = sum([sys.getsizeof(i) for i in user_item_matrix_train.data])/1024**3 
print('matrix size sparse: ', size_gb_sparse)

# создадим ALS-модель. Для примера возьмём количество латентных факторов для матриц $P, Q$, равным 50
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=0)
als_model.fit(user_item_matrix_train) 

def get_recommendations_als(user_item_matrix, model, user_id, user_encoder, item_encoder, include_seen=True, n=5):
    """
    Возвращает отранжированные рекомендации для заданного пользователя
    """
    user_id_enc = user_encoder.transform([user_id])[0]
    recommendations = model.recommend(
         user_id_enc, 
         user_item_matrix[user_id_enc], 
         filter_already_liked_items=not include_seen,
         N=n)
    recommendations = pd.DataFrame({"item_id_enc": recommendations[0], "score": recommendations[1]})
    recommendations["item_id"] = item_encoder.inverse_transform(recommendations["item_id_enc"])
    
    return recommendations 

# получаем список всех возможных user_id (перекодированных)
user_ids_encoded = range(len(user_encoder.classes_))

# получаем рекомендации для всех пользователей
als_recommendations = als_model.recommend(
    user_ids_encoded, 
    user_item_matrix_train[user_ids_encoded], 
    filter_already_liked_items=False, N=100) 

# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(), 
    "score": als_scores.tolist()})
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")

# получаем изначальные идентификаторы
als_recommendations["user_id"] = user_encoder.inverse_transform(als_recommendations["user_id_enc"])
als_recommendations["item_id"] = item_encoder.inverse_transform(als_recommendations["item_id_enc"])
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"]) 

# сохраним полученные рекомендации в файл
als_recommendations = als_recommendations[["user_id", "item_id", "score"]]
als_recommendations.to_parquet("als_recommendations.parquet") 

43304
matrix size:  17
matrix size sparse:  0.26370687410235405


C:\work\yandex\source\sprint4\env_recsys_start\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\work\yandex\source\sprint4\env_recsys_start\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 50/50 [01:04<00:00,  1.29s/it]


### Метрики
Score от ALS не лежат на той же шкале, что и пользовательские оценки. Сравнивать исходные и новые оценки напрямую — некорректно. Поэтому посчитать метрики MAE, RMSE проблематично. Вместо них можно использовать метрики ранжирования. Они сравнивают не абсолютные значения рейтингов и их оценок, а соответствие порядков. Метрики ранжирования покажут, насколько порядок рекомендаций по убыванию score соответствует порядку объектов по убыванию пользовательских оценок. 
На практике часто используют метрику NDCG, она принимает значение от 0 (предлагаемый порядок никак не соответствует истинному) до 1 (предлагаемый порядок в точности соответствует истинному). 
Подробнее о NDCG можно прочитать на MachineLearningInterview.com.


In [11]:
#Для удобства оценки добавим в датафрейм с рекомендациями истинные оценки из тестовой выборки:

als_recommendations = (
    als_recommendations
    .merge(events_test[["user_id", "item_id", "rating"]]
               .rename(columns={"rating": "rating_test"}), 
           on=["user_id", "item_id"], how="left")
)


#Подсчитать метрику NDCG для одного пользователя поможет готовая реализация из scikit-learn
import sklearn.metrics

def compute_ndcg(rating: pd.Series, score: pd.Series, k):

    """ подсчёт ndcg
    rating: истинные оценки
    score: оценки модели
    k: количество айтемов (по убыванию score) для оценки, остальные - отбрасываются
    """
    
    # если кол-во объектов меньше 2, то NDCG - не определена
    if len(rating) < 2:
        return np.nan

    ndcg = sklearn.metrics.ndcg_score(np.asarray([rating.to_numpy()]), np.asarray([score.to_numpy()]), k=k)

    return ndcg 

# Умея считать NDCG для одного пользователя, посчитаем данную метрику, например, для 
# k=5 для всех пользователей из тестовой выборки. 
# В результате каждому пользователю будет соответствовать одно значение NDCG@5. Запись “NDCG@5” означает, что метрика NDCG считается для пяти объектов. 

rating_test_idx = ~als_recommendations["rating_test"].isnull()
ndcg_at_5_scores = als_recommendations[rating_test_idx].groupby("user_id").apply(lambda x: compute_ndcg(x["rating_test"], x["score"], k=5)) 

print(round(ndcg_at_5_scores.mean(), 2))

# Оцените, для какой доли пользователей удалось посчитать метрику NDCG
n_test_users = events_test["user_id"].nunique()
n_with_ndcg = ndcg_at_5_scores.notna().sum()
share = n_with_ndcg / n_test_users
print(round(share, 2))        # 0.14
print(f"{share:.2%}") 

# используя метод  similar_items, получите и оцените рекомендации для нескольких айтемов. Проанализируйте адекватность результатов.
items_meta = items[["item_id", "author", "title", "genre_and_votes"]]
def show_similar_items(item_id, n=5):
    item_id_enc = item_encoder.transform([item_id])[0]
    similar_enc, scores = als_model.similar_items(item_id_enc, N=n + 1)
    similar = pd.DataFrame({"item_id_enc": similar_enc, "score": scores})
    similar["item_id"] = item_encoder.inverse_transform(similar["item_id_enc"])
    similar = similar.merge(items_meta, on="item_id", how="left")
    similar = similar[similar["item_id"] != item_id].head(n)  # убираем саму книгу
    source = items_meta[items_meta["item_id"] == item_id].iloc[0]
    print("=== Исходная книга ===")
    print(f"{source['author']} — {source['title']}")
    print(f"Жанры: {source['genre_and_votes']}")
    print("\n=== Похожие (similar_items) ===")
    print(similar[["author", "title", "score", "genre_and_votes"]].to_string(index=False))
    same_author_share = (similar["author"] == source["author"]).mean()
    print(f"\nДоля рекомендаций того же автора: {same_author_share:.0%}")
    return source, similar
# несколько айтемов: популярный + 2 случайных из train
sample_item_ids = [
    top_k_pop_items["item_id"].iloc[0],
    events_train["item_id"].sample(1, random_state=7).iloc[0],
    events_train["item_id"].sample(1, random_state=21).iloc[0],
]
results = []
for item_id in sample_item_ids:
    source, similar = show_similar_items(item_id, n=5)
    results.append({
        "item_id": item_id,
        "same_author_share": (similar["author"] == source["author"]).mean(),
        "avg_score": similar["score"].mean(),
    })
    print("\n" + "=" * 70 + "\n")
eval_df = pd.DataFrame(results)
print("Сводка по нескольким айтемам:")
print(eval_df.to_string(index=False))
print(f"\nСредняя доля совпадения автора: {eval_df['same_author_share'].mean():.0%}")

0.98
0.14
13.99%
=== Исходная книга ===
Andy Weir — The Martian
Жанры: {'Science Fiction': 11966, 'Fiction': 8430}

=== Похожие (similar_items) ===
            author                                                                      title    score                                                                                                          genre_and_votes
       Xavier Saer                                                             Bleeding Heart 0.797088                                                                 {'Spirituality': 1, 'Self Help-Personal Development': 1}
William Lane Craig                                               Hard Questions, Real Answers 0.765234 {'Philosophy': 11, 'Nonfiction': 10, 'Religion-Christianity': 10, 'Religion': 9, 'Religion-Theology': 8, 'Christian': 7}
       Paul Bogard El fin de la Oscuridad. El ocaso de la noche en una era de luz artificial. 0.755132  {'Nonfiction': 186, 'Science': 109, 'Environment-Nature': 49, 'Environment':

# === Базовые подходы: контентные рекомендации

In [12]:
#составьте список жанров с долями голосов по ним в genres

def get_genres(items):

    """ 
    извлекает список жанров по всем книгам, 
    подсчитывает долю голосов по каждому их них
    """
    
    genres_counter = {}
    
    for k, v, in items.iterrows():
        genre_and_votes =  v["genre_and_votes_dict"]
        if genre_and_votes is None or not isinstance(genre_and_votes, dict):
            continue
        for genre, votes in genre_and_votes.items():
            if votes is None:
                continue
            # увеличиваем счётчик жанров
            try:
                genres_counter[genre] += votes
            except KeyError:
                genres_counter[genre] = 0

    genres = pd.Series(genres_counter, name="votes")
    genres = genres.to_frame()
    genres = genres.reset_index().rename(columns={"index": "name"})
    genres.index.name = "genre_id"
    
    return genres
   
genres = get_genres(items)

genres["score"] = genres["votes"] / genres["votes"].sum()
genres.sort_values(by="score", ascending=False).head(10) 

print(genres.sort_values("score", ascending=False).head(10)[["name", "score"]])

                                   name     score
genre_id                                         
24                              Fantasy  0.149651
0                               Fiction  0.139955
36                             Classics  0.074605
19                          Young Adult  0.072027
34                              Romance  0.052926
9                            Nonfiction  0.037957
18        Historical-Historical Fiction  0.033452
21                              Mystery  0.029956
25                      Science Fiction  0.026629
33                   Fantasy-Paranormal  0.018723


In [13]:
def get_item2genre_matrix(genres, items):
    '''
    Функция строит матрицу вида «книга-жанр». Изучите её. Подумайте, что будет соответствовать столбцам матрицы. 
    '''
    genre_names_to_id = genres.reset_index().set_index("name")["genre_id"].to_dict()
    
    # list to build CSR matrix
    genres_csr_data = []
    genres_csr_row_idx = []
    genres_csr_col_idx = []
    
    for item_idx, (k, v) in enumerate(items.iterrows()):
        genre_and_votes = v["genre_and_votes_dict"]
        if genre_and_votes is None or not isinstance(genre_and_votes, dict):
            continue
        for genre_name, votes in genre_and_votes.items():
            if votes is None:
                continue
            if genre_name not in genre_names_to_id:
                continue
            genre_idx = genre_names_to_id[genre_name]
            genres_csr_data.append(int(votes))
            genres_csr_row_idx.append(item_idx)
            genres_csr_col_idx.append(genre_idx)

    genres_csr = scipy.sparse.csr_matrix((genres_csr_data, (genres_csr_row_idx, genres_csr_col_idx)), shape=(len(items), len(genres)))
    # нормализуем, чтобы сумма оценок принадлежности к жанру была равна 1
    genres_csr = sklearn.preprocessing.normalize(genres_csr, norm='l1', axis=1)
    
    return genres_csr 
    
item2genre_matrix = get_item2genre_matrix(genres, items)
print(f"Строки: книги ({item2genre_matrix.shape[0]}), столбцы: жанры ({item2genre_matrix.shape[1]})")
print("Примеры жанров (столбцы):", genres["name"].head(5).tolist())

# Получим матрицу с весами по жанрам для каждой книги:
items = items.sort_values(by="item_id_enc")
all_items_genres_csr = get_item2genre_matrix(genres, items) 

# Аналогичным образом получим матрицу с весами по жанрам для какого-нибудь пользователя, например, для пользователя с идентификатором 1000010. 
user_id = 1000010
user_events = events_train.query("user_id == @user_id")[["item_id", "rating"]]
user_items = items[items["item_id"].isin(user_events["item_id"])]

user_items_genres_csr = get_item2genre_matrix(genres, user_items)
print(user_items_genres_csr.nnz)


Строки: книги (43312), столбцы: жанры (815)
Примеры жанров (столбцы): ['Fiction', 'Womens Fiction-Chick Lit', 'Humor', 'Politics', 'Autobiography-Memoir']
149


In [ ]:
#Если посчитать средние, то фактически получим предпочтения пользователя по жанрам.
# вычислим склонность пользователя к жанрам как среднее взвешенное значение популяции на его оценки книг.

# преобразуем пользовательские оценки из списка в вектор-столбец
user_ratings = user_events["rating"].to_numpy() / 5
user_ratings = np.expand_dims(user_ratings, axis=1)

user_items_genres_weighted = user_items_genres_csr.multiply(user_ratings)

user_genres_scores = np.asarray(user_items_genres_weighted.mean(axis=0)) 

#Можно посмотреть, какие жанры больше всего нравятся пользователю:
user_genres = genres.copy()
user_genres["score"] = np.ravel(user_genres_scores)
user_genres = user_genres[user_genres["score"] > 0].sort_values(by=["score"], ascending=False)

user_genres.head(5) 

from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# вычисляем сходство между вектором пользователя и векторами по книгам
similarity_scores = cosine_similarity(all_items_genres_csr, user_genres_scores)

# преобразуем в одномерный массив
similarity_scores = similarity_scores.flatten()

# получаем индексы top-k (по убыванию значений), по сути, индексы книг (encoded)
k = 5
top_k_indices = np.argsort(similarity_scores)[::-1][:k]

# строим рекомендации
selected_items = items[items["item_id_enc"].isin(top_k_indices)]

with pd.option_context("max_colwidth", 100):
   display(selected_items[["author", "title", "genre_and_votes"]])

#Преобладающий жанр можно получить, агрегировав голоса из genre_and_votes_dict по всем рекомендованным книгам. Добавьте после selected_items
rec_genres = Counter()
for _, row in selected_items.iterrows():
    for genre, votes in row["genre_and_votes_dict"].items():
        if votes is not None:
            rec_genres[genre] += votes
top_genre, top_votes = rec_genres.most_common(1)[0]
print(f"Преобладающий жанр в рекомендациях: {top_genre} ({top_votes:.0f} голосов)")
print("Топ-5 жанров:")
for genre, votes in rec_genres.most_common(5):
    print(f"  {genre}: {votes:.0f}")


#Получите по алгоритму выше рекомендации для нескольких пользователей, просмотрите их на экране. Подумайте, насколько релевантны и интересны полученные рекомендации пользователям.
#Попробуйте использовать другую меру сходства для получения рекомендаций, например, евклидово расстояние. Проанализируйте, отличаются ли рекомендации от предыдущих. Подумайте почему.
#Задайте собственные предпочтения для наиболее популярных жанров. Посмотрите рекомендации для себя. Прочитали ли бы вы рекомендованные книги?

,author,title,genre_and_votes
80465,G.K. Chesterton,The Napoleon of Notting Hill,"{'Fiction': 166, 'Classics': 88, 'Fantasy': 44, 'Humor': 22, 'Literature': 20}"
1168335,Ray Bradbury,"Dandelion Wine (Green Town, #1)","{'Fiction': 1438, 'Classics': 914, 'Science Fiction': 529, 'Fantasy': 456, 'Young Adult': 212}"
393210,"G.K. Chesterton, Jonathan Lethem",The Man Who Was Thursday: A Nightmare,"{'Fiction': 1257, 'Classics': 929, 'Mystery': 469, 'Fantasy': 293, 'Philosophy': 156, 'Literatur..."
2244467,Samuel Butler,"Erewhon (Erewhon , #1)","{'Fiction': 162, 'Classics': 139, 'Science Fiction': 60, 'Fantasy': 55}"
39408,"Paulo Coelho, Alan R. Clarke, James Noel Smith",The Alchemist,"{'Fiction': 14023, 'Classics': 5787, 'Fantasy': 3289, 'Philosophy': 2759}"


Преобладающий жанр в рекомендациях: Fiction (17046 голосов)
Топ-5 жанров:
  Fiction: 17046
  Classics: 7857
  Fantasy: 4137
  Philosophy: 2915
  Science Fiction: 589


# === Базовые подходы: валидация

In [ ]:
def process_events_recs_for_binary_metrics(events_train, events_test, recs, top_k=None):

    """
    размечает пары <user_id, item_id> для общего множества пользователей признаками
    - gt (ground truth)
    - pr (prediction)
    top_k: расчёт ведётся только для top k-рекомендаций
    """

    events_test["gt"] = True
    common_users = set(events_test["user_id"]) & set(recs["user_id"])

    print(f"Common users: {len(common_users)}")
    
    events_for_common_users = events_test[events_test["user_id"].isin(common_users)].copy()
    recs_for_common_users = recs[recs["user_id"].isin(common_users)].copy()

    recs_for_common_users = recs_for_common_users.sort_values(["user_id", "score"], ascending=[True, False])

    # оставляет только те item_id, которые были в events_train, 
    # т. к. модель не имела никакой возможности давать рекомендации для новых айтемов
    events_for_common_users = events_for_common_users[events_for_common_users["item_id"].isin(events_train["item_id"].unique())]

    if top_k is not None:
        recs_for_common_users = recs_for_common_users.groupby("user_id").head(top_k)
    
    events_recs_common = events_for_common_users[["user_id", "item_id", "gt"]].merge(
        recs_for_common_users[["user_id", "item_id", "score"]], 
        on=["user_id", "item_id"], how="outer")    

    events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
    events_recs_common["pr"] = ~events_recs_common["score"].isnull()
    
    events_recs_common["tp"] = events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fp"] = ~events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fn"] = events_recs_common["gt"] & ~events_recs_common["pr"]

    return events_recs_common

events_recs_for_binary_metrics = process_events_recs_for_binary_metrics(
  events_train,
    events_test, 
    als_recommendations, 
    top_k=5) 

# Мы получили разметку случившихся в будущем событий (events_test) и рекомендаций (als_recommendations: то есть событий, спрогнозированных как случившиеся в будущем). На их основе можно посчитать precision и recall по вышеприведённым формулам. Ниже дана заготовка функции, которая считает эти метрики. На вход ей подаётся результат выполнения функции process_events_recs_for_binary_metrics


def compute_cls_metrics(events_recs_for_binary_metric):
    
    groupper = events_recs_for_binary_metric.groupby("user_id")

    # precision = tp / (tp + fp)
    precision = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fp"].sum())
    precision = precision.fillna(0).mean()
    
    # recall = tp / (tp + fn)
    recall = groupper["tp"].sum() / (groupper["tp"].sum() + groupper["fn"].sum())
    recall = recall.fillna(0).mean()

    return precision, recall 

precision_at_5, recall_at_5 = compute_cls_metrics(events_recs_for_binary_metrics)

print(f"precision@5: {precision_at_5:.3f}")
print(f"recall@5: {recall_at_5:.3f}")

#Посчитайте метрики precision@10, recall@10. Сравните их значения со значениями для precision@5, recall@5. Подумайте о причинах таких отличий.
events_recs_for_binary_metrics = process_events_recs_for_binary_metrics(
  events_train,
    events_test, 
    als_recommendations, 
    top_k=10) 

precision_at_10, recall_at_10 = compute_cls_metrics(events_recs_for_binary_metrics)

print(f"precision@10: {precision_at_10:.3f}")
print(f"recall@10: {recall_at_10:.3f}")


Common users: 123223
precision@5: 0.008
recall@5: 0.014
Common users: 123223
precision@10: 0.009
recall@10: 0.031


In [16]:
#Для рекомендаций, сохранённых в переменной als_recommendations, посчитайте покрытие по объектам согласно формуле выше. При этом используйте весь топ-100 рекомендаций.
# расчёт покрытия по объектам
als_top100 = (
    als_recommendations
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")
    .head(100)
)
cov_items = als_top100["item_id"].nunique() / len(items)
print("cover_items 100: ", f"{cov_items:.2f}") 

# разметим каждую рекомендацию признаком read
events_train["read"] = True
als_recommendations = als_recommendations.merge(
    events_train[["user_id", "item_id", "read"]],
    on=["user_id", "item_id"], 
    how="left")
als_recommendations["read"] = als_recommendations["read"].fillna(False).astype("bool")

# проставим ранги
als_recommendations = als_recommendations.sort_values(["user_id", "score"], ascending=[True, False])
als_recommendations["rank"] = als_recommendations.groupby("user_id").cumcount() + 1

# посчитаем novelty по пользователям
novelty_5 = (1-als_recommendations.query("rank <= 5").groupby("user_id")["read"].mean())

# посчитаем средний novelty
novelty_5_mean = novelty_5.mean()
print(f"Novelty@5: {novelty_5_mean:.2f}")

cover_items 100:  0.09
Novelty@5: 0.61


# === Двухстадийный подход: метрики

In [17]:
# задаём точку разбиения
split_date_for_labels = pd.to_datetime("2017-09-15").date()

split_date_for_labels_idx = events_test["started_at"] < split_date_for_labels
events_labels = events_test[split_date_for_labels_idx].copy()
events_test_2 = events_test[~split_date_for_labels_idx].copy() 

print(f"Уникальных пользователей в events_labels: {events_labels['user_id'].nunique()}")

Уникальных пользователей в events_labels: 99849


### Подготовка кандидатов для обучения
Подготовим список кандидатов для обучения ранжирующей модели. В качестве кандидатогенераторов возьмём ALS и контентную модель на основе жанровых предпочтений, известных нам из прошлых уроков. Рекомендации от них были заранее подготовлены и сохранены в файлах als_recommendations.parquet и content_recommendations.parquet в директории candidates/training. Подготовка заключается в объединении списков рекомендаций по совпадению user_id, item_id. 

In [18]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# загружаем рекомендации от двух базовых генераторов
als_recommendations = pd.read_parquet("candidates/training/als_recommendations.parquet")
content_recommendations = pd.read_parquet("candidates/training/content_recommendations.parquet")

#Что делает код:
#on=["user_id", "item_id"] — объединение по паре пользователь + книга;
#how="outer" — в candidates попадают все пары из ALS и контентных рекомендаций (даже если книга есть только в одном источнике);
#len(candidates) — число строк в итоговом списке кандидатов.

candidates = pd.merge(
    als_recommendations[["user_id", "item_id", "score"]].rename(columns={"score": "als_score"}),
    content_recommendations[["user_id", "item_id", "score"]].rename(columns={"score": "cnt_score"}),
    on=["user_id", "item_id"],
    how="outer") 

print(f"Записей в candidates: {len(candidates)}")


Записей в candidates: 82993094


In [19]:
# добавляем таргет к кандидатам со значением:
# — 1 для тех item_id, которые пользователь прочитал
# — 0, для всех остальных 

events_labels["target"] = 1
candidates = candidates.merge(
    events_labels[["user_id", "item_id", "target"]], 
    on=["user_id", "item_id"],
    how="left",)
candidates["target"] = candidates["target"].fillna(0).astype("int")

# в кандидатах оставляем только тех пользователей, у которых есть хотя бы один положительный таргет
candidates_to_sample = candidates.groupby("user_id").filter(lambda x: x["target"].sum() > 0)

# для каждого пользователя оставляем только 4 негативных примера
negatives_per_user = 4
candidates_for_train = pd.concat([
    candidates_to_sample.query("target == 1"),
    candidates_to_sample.query("target == 0")
        .groupby("user_id")
        .apply(lambda x: x.sample(negatives_per_user, random_state=0))
    ]) 
print(f"Записей в candidates_for_train: {len(candidates_for_train)}")


Записей в candidates_for_train: 213708


# === Двухстадийный подход: модель

### Подготовка кандидатов для рекомендаций
Представим, что натренированная модель используется только некоторое время спустя, когда уже появились новые рекомендации (кандидаты) от базовых генераторов, обученных на объединении событий из events_train и events_label. Иными словами, когда события из events_label уже стали частью тренировочного набора данных. Эти новые рекомендации были заранее подготовлены и сохранены в файлах als_recommendations.parquet и content_recommendations.parquet в директории candidates/inference. Используем их для составления нового списка кандидатов candidates_to_rank, который понадобится готовой ранжирующей модели. 

In [20]:
from catboost import CatBoostClassifier, Pool

# задаём имена колонок признаков и таргета
features = ['als_score', 'cnt_score']
target = 'target'

# Create the Pool object
train_data = Pool(
    data=candidates_for_train[features], 
    label=candidates_for_train[target])

# инициализируем модель CatBoostClassifier
cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_seed=0
)

# тренируем модель
cb_model.fit(train_data) 

# загружаем рекомендации от двух базовых генераторов
als_recommendations_2 = pd.read_parquet("candidates/inference/als_recommendations.parquet")
content_recommendations_2 = pd.read_parquet("candidates/inference/content_recommendations.parquet")

candidates_to_rank = pd.merge(
    als_recommendations_2[["user_id", "item_id", "score"]].rename(columns={"score": "als_score"}),
    content_recommendations_2[["user_id", "item_id", "score"]].rename(columns={"score": "cnt_score"}),
    on=["user_id", "item_id"],
    how="outer",)

# оставляем только тех пользователей, что есть в тестовой выборке, для экономии ресурсов
candidates_to_rank = candidates_to_rank[candidates_to_rank["user_id"].isin(events_test_2["user_id"].drop_duplicates())]
print(f"Записей в candidates_to_rank: {len(candidates_to_rank)}")

0:	learn: 0.6526228	total: 173ms	remaining: 2m 52s
100:	learn: 0.5118920	total: 1.34s	remaining: 11.9s
200:	learn: 0.5112252	total: 2.45s	remaining: 9.74s
300:	learn: 0.5105772	total: 3.58s	remaining: 8.32s
400:	learn: 0.5100526	total: 4.7s	remaining: 7.01s
500:	learn: 0.5095749	total: 5.85s	remaining: 5.83s
600:	learn: 0.5091867	total: 6.97s	remaining: 4.62s
700:	learn: 0.5088334	total: 8.05s	remaining: 3.44s
800:	learn: 0.5085141	total: 9.14s	remaining: 2.27s
900:	learn: 0.5081793	total: 10.3s	remaining: 1.13s
999:	learn: 0.5078908	total: 11.4s	remaining: 0us
Записей в candidates_to_rank: 14517152


### Ранжирование кандидатов для рекомендаций
Применим обученную ранжирующую модель к кандидатам для рекомендаций. Таргет уже не нужен, поскольку мы будем применять модель в режиме инференса.

In [23]:
inference_data = Pool(data=candidates_to_rank[features])
predictions = cb_model.predict_proba(inference_data)

candidates_to_rank["cb_score"] = predictions[:, 1]

# для каждого пользователя проставляем rank, начиная с 1 — это максимальный cb_score
candidates_to_rank = candidates_to_rank.sort_values(["user_id", "cb_score"], ascending=[True, False])

max_recommendations_per_user = 100
candidates_to_rank["rank"] = candidates_to_rank.groupby("user_id").cumcount() + 1

final_recommendations = candidates_to_rank.query("rank <= @max_recommendations_per_user")

final_recommendations[["user_id", "item_id", "rank"]].to_parquet("final_recommendations.parquet")

print("final_recommendations", final_recommendations)

final_recommendations           user_id   item_id  als_score  cnt_score  cb_score  rank
347       1000003     49628   0.446143   0.906649  0.605896     1
300       1000003   7260188   1.129979        NaN  0.509745     2
301       1000003   6148028   1.123475        NaN  0.509745     3
302       1000003   2767052   1.112699        NaN  0.509745     4
320       1000003     43641   0.617602        NaN  0.476134     5
...           ...       ...        ...        ...       ...   ...
43058096  1430580   6314763   0.016404        NaN  0.232464    96
43058094  1430580   4327066   0.016494   0.966518  0.230046    97
43058097  1430580  11710373   0.016035        NaN  0.221189    98
43058098  1430580      7445   0.015793        NaN  0.221189    99
43058099  1430580  17802724   0.015771        NaN  0.221189   100

[7519400 rows x 6 columns]


### Валидация
Для валидации применимы всё те же метрики из предыдущих уроков. Считать их можно, как мы уже упоминали, как для базовых генераторов, так и для ранжирующей модели.
Подобным же образом при валидации можно визуально просматривать как рекомендации от базовых моделей, так и финальные. Такой ручной просмотр по пользователям может подсказать, какой смысловой вклад вносят те или иные кандидатогенераторы в итоговые рекомендации.

In [26]:
events_inference = pd.concat([events_train, events_labels])

cb_events_recs_for_binary_metrics_5 = process_events_recs_for_binary_metrics(
    events_inference,
    events_test_2,
    final_recommendations.rename(columns={"cb_score": "score"}), 
    top_k=5)

cb_precision_5, cb_recall_5 = compute_cls_metrics(cb_events_recs_for_binary_metrics_5)

recall_rounded = round(cb_recall_5, 3)
print(f"precision: {cb_precision_5:.3f}, recall: {recall_rounded:.3f}") 

Common users: 75194
precision: 0.006, recall: 0.015


# === Двухстадийный подход: построение признаков

признаки книги
age = 2018 - publication_year
merge age и average_rating в candidates_for_train и candidates_to_rank

In [27]:
items["age"] = 2018 - items["publication_year"]
invalid_age_idx = items["age"] < 0
items.loc[invalid_age_idx, "age"] = np.nan
items["age"] = items["age"].astype("float")

candidates_for_train = candidates_for_train.merge(
    items[["item_id", "age", "average_rating"]],
    on="item_id",
    how="left",
)

candidates_to_rank = candidates_to_rank.merge(
    items[["item_id", "age", "average_rating"]],
    on="item_id",
    how="left",
)

print(f"Медианный возраст книги в candidates_to_rank: {candidates_to_rank['age'].median()}")

Медианный возраст книги в candidates_to_rank: 7.0


признаки пользователя
функция get_user_features (reading_years, books_read, rating_avg, rating_std, books_per_year)
merge в candidates_for_train по events_train
merge в candidates_to_rank по events_inference (train + labels)

In [34]:
def get_user_features(events):
    """ Строит пользовательские признаки """

    user_features = events.groupby("user_id").agg(
        reading_years=("started_at", lambda x: (x.max() - x.min()).days / 365.25),
        books_read=("item_id", "count"),
        rating_avg=("rating", "mean"),
        rating_std=("rating", "std"))

    user_features["books_per_year"] = user_features["books_read"] / user_features["reading_years"]

    return user_features

user_feature_cols = [
    "reading_years", "books_read", "rating_avg", "rating_std", "books_per_year",
]

def drop_user_feature_cols(df):
    cols = [
        c for c in df.columns
        if c in user_feature_cols or any(c.startswith(f"{col}_") for col in user_feature_cols)
    ]
    return df.drop(columns=cols, errors="ignore")

user_features_for_train = get_user_features(events_train)
candidates_for_train = drop_user_feature_cols(candidates_for_train)
candidates_for_train = candidates_for_train.merge(user_features_for_train, on="user_id", how="left")

events_inference = pd.concat([events_train, events_labels])
events_inference = events_inference[events_inference["user_id"].isin(events_test["user_id"].drop_duplicates())]

user_features_for_ranking = get_user_features(events_inference)
candidates_to_rank = drop_user_feature_cols(candidates_to_rank)
candidates_to_rank = candidates_to_rank.merge(user_features_for_ranking, on="user_id", how="left")

print(f"Медиана books_read в candidates_for_train: {candidates_for_train['books_read'].median()}")

Медиана books_read в candidates_for_train: 32.0


жанровые признаки
топ-10 жанров + genre_others
merge жанровости книг в items
функция get_user_genres и merge в оба датафрейма кандидатов
печать медианы Romance (ожидается 0.04)


In [30]:
genres_top_k = 10
genres_top_idx = genres.sort_values("votes", ascending=False).head(genres_top_k).index
genres_others_idx = list(set(genres.index) - set(genres_top_idx))

genres_top_columns = [f"genre_{id}" for id in genres_top_idx]
genres_others_column = "genre_others"
genre_columns = genres_top_columns + [genres_others_column]

item_genres = (
    pd.concat([
        pd.DataFrame(
            all_items_genres_csr[:, genres_top_idx].toarray(),
            columns=genres_top_columns,
        ),
        pd.DataFrame(
            all_items_genres_csr[:, genres_others_idx].sum(axis=1),
            columns=[genres_others_column],
        ),
    ], axis=1)
    .reset_index()
    .rename(columns={"index": "item_id_enc"})
)

items = items.drop(
    columns=[c for c in items.columns if c.startswith("genre_")],
    errors="ignore",
)
items = items.merge(item_genres, on="item_id_enc", how="left")

def get_user_genres(events, items, item_genre_columns):
    user_genres = (
        events
        .merge(items[["item_id"] + item_genre_columns], on="item_id", how="left")
        .groupby("user_id")[item_genre_columns]
        .mean()
    )
    return user_genres

user_genres_for_train = get_user_genres(events_train, items, genre_columns)

genre_cols = [c for c in candidates_for_train.columns if c.startswith("genre_")]
candidates_for_train = candidates_for_train.drop(columns=genre_cols, errors="ignore")
candidates_for_train = candidates_for_train.merge(user_genres_for_train, on="user_id", how="left")

user_genres_for_ranking = get_user_genres(events_inference, items, genre_columns)

genre_cols = [c for c in candidates_to_rank.columns if c.startswith("genre_")]
candidates_to_rank = candidates_to_rank.drop(columns=genre_cols, errors="ignore")
candidates_to_rank = candidates_to_rank.merge(user_genres_for_ranking, on="user_id", how="left")

romance_col = f"genre_{genres[genres['name'] == 'Romance'].index[0]}"
print(round(candidates_for_train[romance_col].median(), 2))

0.04


ячейка обучения CatBoost с полным списком features и исправлением cb_score.

 код обучения CatBoost с полным списком признаков

In [37]:
from catboost import CatBoostClassifier, Pool
features = ['als_score', 'cnt_score',
    'age', 'average_rating', 'reading_years', 'books_read',
    'rating_avg', 'rating_std',
    'books_per_year'] + genre_columns
target = 'target'
train_data = Pool(
    data=candidates_for_train[features],
    label=candidates_for_train[target])
cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_seed=0
)
cb_model.fit(train_data)

inference_data = Pool(data=candidates_to_rank[features])
predictions = cb_model.predict_proba(inference_data)
candidates_to_rank = candidates_to_rank.drop(columns=["cb_score", "rank"], errors="ignore")
candidates_to_rank["cb_score"] = predictions[:, 1]
candidates_to_rank = candidates_to_rank.sort_values(["user_id", "cb_score"], ascending=[True, False])
max_recommendations_per_user = 100

candidates_to_rank["rank"] = candidates_to_rank.groupby("user_id").cumcount() + 1
final_recommendations = candidates_to_rank[candidates_to_rank["rank"] <= max_recommendations_per_user]
final_recommendations[["user_id", "item_id", "rank"]].to_parquet("final_recommendations_feat.parquet")

print(f"Пользователей: {final_recommendations['user_id'].nunique()}")
print(final_recommendations.head())


0:	learn: 0.6484916	total: 29.3ms	remaining: 29.3s
100:	learn: 0.4659847	total: 1.57s	remaining: 14s
200:	learn: 0.4575702	total: 3.09s	remaining: 12.3s
300:	learn: 0.4516846	total: 4.59s	remaining: 10.7s
400:	learn: 0.4470707	total: 6.09s	remaining: 9.1s
500:	learn: 0.4428381	total: 7.58s	remaining: 7.55s
600:	learn: 0.4390354	total: 9.1s	remaining: 6.04s
700:	learn: 0.4354365	total: 10.6s	remaining: 4.53s
800:	learn: 0.4320561	total: 12.2s	remaining: 3.02s
900:	learn: 0.4288497	total: 13.7s	remaining: 1.5s
999:	learn: 0.4258418	total: 15.2s	remaining: 0us
Пользователей: 75194
    user_id  item_id  als_score  cnt_score   age  average_rating  genre_24  \
3   1000003  2767052   1.112699        NaN  10.0            4.34  0.068109   
1   1000003  7260188   1.129979        NaN   8.0            4.03  0.068109   
2   1000003  6148028   1.123475        NaN   9.0            4.30  0.068109   
98  1000003  9361589   1.060634        NaN   7.0            4.03  0.068109   
47  1000003     7445   0.

In [38]:
uid = 1353637
candidates_to_rank[candidates_to_rank["user_id"] == uid].sort_values(
    "cb_score", ascending=False
).head(5)[["item_id", "cb_score"]]

,item_id,cb_score
11913122,28187230,0.692986
11913106,5,0.681202
11913127,27161156,0.676046
11913118,27833670,0.657702
11913134,30555488,0.640738


In [36]:
final_recommendations[["user_id", "item_id", "rank"]].to_parquet("final_recommendations_feat.parquet")
print(
    f"Сохранено: {len(final_recommendations)} строк, "
    f"{final_recommendations['user_id'].nunique()} пользователей"
)

Сохранено: 7519400 строк, 75194 пользователей
